# 00 · Exploración y perfil inicial de las fuentes (Landing)

**Proyecto:** Cloud Provider Analytics · Big Data ITBA 2C 2026 · **Entrega 1**

Objetivo: evidencia mínima de lectura y exploración con PySpark (consigna §5.2 ítem 12). El notebook **solo lee** `datalake/landing/` (inmutable) y escribe un resumen en `evidence/entrega1/`. No implementa Bronze/Silver/Gold.

**Cómo correrlo**
- *Google Colab:* `!git clone <url-del-repo>` → `%cd <repo>/notebooks` → ejecutar todo. La primera celda instala `pyspark==3.5.3` si no está.
- *Local:* Python 3.10–3.12, Java 17 y `pip install pyspark==3.5.3`; ejecutar desde `notebooks/`.
- La ruta de Landing se puede cambiar con la variable de entorno `LANDING_DIR`.

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"])

import os, json, glob
from datetime import datetime, timezone
from pyspark.sql import SparkSession, functions as F

LANDING_DIR = os.environ.get("LANDING_DIR", "../datalake/landing")
EVIDENCE_DIR = os.environ.get("EVIDENCE_DIR", "../evidence/entrega1")
os.makedirs(EVIDENCE_DIR, exist_ok=True)

spark = (SparkSession.builder.appName("cpa-00-exploracion").master("local[*]")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "8")   # dataset chico: evitar 200 tareas vacías
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| Landing:", LANDING_DIR)
resumen = {"generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "spark": spark.version}

Spark 3.5.3 | Landing: ../datalake/landing


## 1. Inventario de Landing (archivos y tamaños)

In [2]:
files = sorted(glob.glob(os.path.join(LANDING_DIR, "*.csv")))
events_files = sorted(glob.glob(os.path.join(LANDING_DIR, "usage_events_stream", "*.jsonl")))
inv = [(os.path.basename(f), os.path.getsize(f)) for f in files]
inv.append(("usage_events_stream/*.jsonl (%d archivos)" % len(events_files), sum(os.path.getsize(f) for f in events_files)))
for name, size in inv:
    print(f"{name:45s} {size/1024:10.1f} KB")
total = sum(s for _, s in inv)
print(f"{'TOTAL':45s} {total/1024/1024:10.2f} MB")
resumen["landing_bytes"] = total
resumen["archivos_eventos"] = len(events_files)

billing_monthly.csv                                 15.5 KB
customers_orgs.csv                                   7.6 KB
marketing_touches.csv                               99.6 KB
nps_surveys.csv                                      3.9 KB
resources.csv                                       35.8 KB
support_tickets.csv                                 69.3 KB
users.csv                                           69.0 KB
usage_events_stream/*.jsonl (120 archivos)       12624.5 KB
TOTAL                                              12.62 MB


## 2. Fuentes batch (CSV)

Lectura **sin inferir tipos** (todo `string`): en exploración interesa ver el dato crudo tal como llega. Spark convierte los campos vacíos en `null`. Para cada columna: nulos, valores distintos y una muestra.

In [3]:
CSV_SOURCES = ["customers_orgs", "users", "resources", "support_tickets",
               "marketing_touches", "nps_surveys", "billing_monthly"]
dfs = {}
resumen["csv"] = {}
for name in CSV_SOURCES:
    df = (spark.read.option("header", True).option("multiLine", False)
          .option("escape", '"').csv(os.path.join(LANDING_DIR, f"{name}.csv")))
    dfs[name] = df
    n = df.count()
    nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).first().asDict()
    distinct = df.select([F.countDistinct(c).alias(c) for c in df.columns]).first().asDict()
    resumen["csv"][name] = {"filas": n, "columnas": df.columns, "nulos": nulls, "distintos": distinct}
    print(f"\n=== {name}: {n} filas, {len(df.columns)} columnas")
    for c in df.columns:
        muestra = [r[0] for r in df.select(c).where(F.col(c).isNotNull()).distinct().limit(5).collect()]
        print(f"  {c:22s} nulos={nulls[c]:5d}  distintos={distinct[c]:5d}  ej={muestra}")


=== customers_orgs: 80 filas, 11 columnas


  org_id                 nulos=    0  distintos=   80  ej=['org_hv3a3zmf', 'org_41ibljh7', 'org_sg65kxvf', 'org_om5igqpk', 'org_rixa11dp']


  org_name               nulos=    0  distintos=   80  ej=['Omega Tech 5', 'Alpha Labs 21', 'Omega Labs 38', 'Delta Cloud 43', 'Omega Cloud 46']


  industry               nulos=    0  distintos=   10  ej=['Energy', 'Government', 'Media', 'Healthcare', 'Gaming']


  hq_region              nulos=    0  distintos=    7  ej=['us-west', 'eu-central', 'us-east', 'ap-northeast', 'sa-east']


...


=== billing_monthly: 240 filas, 8 columnas
  invoice_id             nulos=    0  distintos=  240  ej=['inv_n5n5ehq4', 'inv_yohcknew', 'inv_4osu2ma9', 'inv_n2tvdnhb', 'inv_427u1nvm']


  org_id                 nulos=    0  distintos=   80  ej=['org_hv3a3zmf', 'org_41ibljh7', 'org_sg65kxvf', 'org_om5igqpk', 'org_rixa11dp']
  month                  nulos=    0  distintos=    3  ej=['2025-06-01', '2025-08-01', '2025-07-01']


  subtotal               nulos=    0  distintos=  240  ej=['1851.84', '1033.34', '719.67', '345.59', '730.7']
  credits                nulos=  137  distintos=  100  ej=['0.62', '7.03', '12.45', '46.81', '1.21']


  taxes                  nulos=    0  distintos=  240  ej=['121.82', '172.07', '160.5', '143.15', '93.32']
  currency               nulos=    0  distintos=    3  ej=['ARS', 'USD', 'EUR']


  exchange_rate_to_usd   nulos=    0  distintos=  208  ej=['1.03282', '1.00132', '0.93308', '0.96453', '1.10725']


### 2.1 Unicidad de claves naturales e integridad referencial

In [4]:
PK = {"customers_orgs": ["org_id"], "users": ["user_id"], "resources": ["resource_id"],
      "support_tickets": ["ticket_id"], "marketing_touches": ["touch_id"],
      "billing_monthly": ["invoice_id"], "nps_surveys": ["org_id", "survey_date"]}
orgs = dfs["customers_orgs"].select("org_id").distinct()
resumen["claves"] = {}
for name, keys in PK.items():
    df = dfs[name]
    dups = df.groupBy(*keys).count().where("count > 1").count()
    huerfanos = df.join(orgs, "org_id", "left_anti").count() if name != "customers_orgs" else 0
    resumen["claves"][name] = {"pk": keys, "claves_duplicadas": dups, "org_id_huerfanos": huerfanos}
    print(f"{name:18s} PK={keys}  claves duplicadas={dups}  org_id huérfanos={huerfanos}")

dup_org_month = dfs["billing_monthly"].groupBy("org_id", "month").count().where("count > 1").count()
print("billing_monthly: (org_id, month) duplicados =", dup_org_month)
resumen["billing_org_month_duplicados"] = dup_org_month

customers_orgs     PK=['org_id']  claves duplicadas=0  org_id huérfanos=0


users              PK=['user_id']  claves duplicadas=0  org_id huérfanos=0


resources          PK=['resource_id']  claves duplicadas=0  org_id huérfanos=0


support_tickets    PK=['ticket_id']  claves duplicadas=0  org_id huérfanos=0


marketing_touches  PK=['touch_id']  claves duplicadas=0  org_id huérfanos=0


billing_monthly    PK=['invoice_id']  claves duplicadas=0  org_id huérfanos=0


nps_surveys        PK=['org_id', 'survey_date']  claves duplicadas=0  org_id huérfanos=0
billing_monthly: (org_id, month) duplicados = 0


### 2.2 Chequeos de dominio (veracidad)

In [5]:
num = lambda c: F.col(c).cast("double")
b = dfs["billing_monthly"]
print("Billing por moneda y rango de exchange_rate_to_usd:")
fx = (b.groupBy("currency").agg(F.count("*").alias("facturas"),
                                F.min(num("exchange_rate_to_usd")).alias("fx_min"),
                                F.max(num("exchange_rate_to_usd")).alias("fx_max"))
        .orderBy("currency"))
fx.show()
print("Meses facturados:", [r[0] for r in b.select("month").distinct().orderBy("month").collect()])

checks = {
    "customers_orgs.nps_score fuera de [-100,100]": dfs["customers_orgs"].where((num("nps_score") < -100) | (num("nps_score") > 100)).count(),
    "nps_surveys.nps_score fuera de [-100,100]": dfs["nps_surveys"].where((num("nps_score") < -100) | (num("nps_score") > 100)).count(),
    "support_tickets.csat fuera de [1,5]": dfs["support_tickets"].where((num("csat") < 1) | (num("csat") > 5)).count(),
    "support_tickets abiertos (resolved_at nulo)": dfs["support_tickets"].where(F.col("resolved_at").isNull()).count(),
    "support_tickets resolved_at < created_at": dfs["support_tickets"].where(F.to_date("resolved_at") < F.to_date("created_at")).count(),
    "billing credits nulos": b.where(F.col("credits").isNull()).count(),
    "billing credits negativos": b.where(num("credits") < 0).count(),
    "billing USD con exchange_rate != 1": b.where((F.col("currency") == "USD") & (num("exchange_rate_to_usd") != 1.0)).count(),
}
for k, v in checks.items():
    print(f"{k:50s} {v}")
resumen["chequeos_dominio"] = checks

print("\nRango de nps_score (escala NPS -100..100):")
for name in ["customers_orgs", "nps_surveys"]:
    r = dfs[name].agg(F.min(num("nps_score")), F.max(num("nps_score"))).first()
    print(f"  {name:15s} min={r[0]}  max={r[1]}")
print("Distribución de csat:")
csat = dfs["support_tickets"].groupBy(num("csat").alias("csat")).count().orderBy("csat")
csat.show()
resumen["csat_distribucion"] = {str(r["csat"]): r["count"] for r in csat.collect()}
resumen["billing_fx"] = [r.asDict() for r in fx.collect()]

t = dfs["support_tickets"]
print("\nTickets por severidad y tasa de SLA breach:")
t.groupBy("severity").agg(F.count("*").alias("tickets"),
                          F.round(F.avg((F.col("sla_breached") == "True").cast("int")), 3).alias("sla_breach_rate")
                         ).orderBy("severity").show()
rangos = {
    "support_tickets.created_at": t.agg(F.min("created_at"), F.max("created_at")).first(),
    "marketing_touches.timestamp": dfs["marketing_touches"].agg(F.min("timestamp"), F.max("timestamp")).first(),
    "nps_surveys.survey_date": dfs["nps_surveys"].agg(F.min("survey_date"), F.max("survey_date")).first(),
    "customers_orgs.signup_date": dfs["customers_orgs"].agg(F.min("signup_date"), F.max("signup_date")).first(),
}
for k, (mn, mx) in rangos.items():
    print(f"Rango {k:30s} {mn} → {mx}")
resumen["rangos_fechas"] = {k: [mn, mx] for k, (mn, mx) in rangos.items()}

Billing por moneda y rango de exchange_rate_to_usd:


+--------+--------+-------+-------+
|currency|facturas| fx_min| fx_max|
+--------+--------+-------+-------+
|     ARS|      51|0.00133|0.00162|
|     EUR|      29| 0.9981|1.19808|
|     USD|     160|0.85463|1.11791|
+--------+--------+-------+-------+



Meses facturados: ['2025-06-01', '2025-07-01', '2025-08-01']


customers_orgs.nps_score fuera de [-100,100]       1
nps_surveys.nps_score fuera de [-100,100]          0
support_tickets.csat fuera de [1,5]                40
support_tickets abiertos (resolved_at nulo)        240
support_tickets resolved_at < created_at           0
billing credits nulos                              137
billing credits negativos                          0
billing USD con exchange_rate != 1                 160

Rango de nps_score (escala NPS -100..100):
  customers_orgs  min=-38.0  max=101.0


  nps_surveys     min=-16.0  max=68.0
Distribución de csat:


+----+-----+
|csat|count|
+----+-----+
|NULL|  254|
| 0.0|   11|
| 1.0|   45|
| 2.0|  127|
| 3.0|  242|
| 4.0|  197|
| 5.0|   95|
| 6.0|   28|
| 7.0|    1|
+----+-----+




Tickets por severidad y tasa de SLA breach:


+--------+-------+---------------+
|severity|tickets|sla_breach_rate|
+--------+-------+---------------+
|critical|     56|          0.036|
|    high|    204|          0.098|
|     low|    412|          0.087|
|  medium|    328|          0.113|
+--------+-------+---------------+



Rango support_tickets.created_at     2025-05-09 → 2025-08-31
Rango marketing_touches.timestamp    2025-05-04 → 2025-08-31
Rango nps_surveys.survey_date        2025-05-24 → 2025-08-31
Rango customers_orgs.signup_date     2025-05-04 → 2025-07-02


## 3. Eventos de uso (`usage_events_stream/*.jsonl`)

Primero se lee con **inferencia** para descubrir el esquema real (en la entrega 2 se usará un `StructType` explícito). Luego se lee como **texto** para detectar el tipo con que llega `value` en el JSON original, porque la inferencia lo unifica a `string`.

In [6]:
# Se pasa la lista explícita de archivos (glob de Python): en Windows sin winutils.exe
# Spark no puede listar directorios locales; en Colab/Linux es equivalente a leer el directorio.
ev = (spark.read.json(events_files)
      .withColumn("source_file", F.element_at(F.split(F.input_file_name(), "/"), -1))
      .withColumn("event_ts", F.to_timestamp("timestamp")))
ev.printSchema()
ev.cache()
n_ev = ev.count()
print("Eventos:", n_ev)

root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)



Eventos: 43200


In [7]:
dq = ev.agg(
    F.countDistinct("event_id").alias("event_id_distintos"),
    F.sum(F.col("event_id").isNull().cast("int")).alias("event_id_nulos"),
    F.sum(F.col("event_ts").isNull().cast("int")).alias("timestamp_no_parseable"),
    # date_format en Spark (zona de sesión = UTC); collect() convertiría a la zona local de Python
    F.date_format(F.min("event_ts"), "yyyy-MM-dd HH:mm:ss").alias("ts_min_utc"),
    F.date_format(F.max("event_ts"), "yyyy-MM-dd HH:mm:ss").alias("ts_max_utc"),
    F.countDistinct(F.to_date("event_ts")).alias("dias_con_eventos"),
    F.sum(F.col("value").isNull().cast("int")).alias("value_nulos"),
    F.sum(F.col("unit").isNull().cast("int")).alias("unit_nulos"),
    F.sum((F.col("value").isNotNull() & F.col("unit").isNull()).cast("int")).alias("unit_nulo_con_value"),
    F.sum((F.col("cost_usd_increment") < -0.01).cast("int")).alias("costo_menor_a_-0.01"),
    F.sum((F.col("cost_usd_increment") < 0).cast("int")).alias("costo_negativo"),
    F.min("cost_usd_increment").alias("costo_min"), F.max("cost_usd_increment").alias("costo_max"),
).first().asDict()
dq["event_id_duplicados"] = n_ev - dq["event_id_distintos"]
q = ev.approxQuantile("cost_usd_increment", [0.5, 0.99, 0.999], 0.0)
dq.update({"costo_p50": q[0], "costo_p99": q[1], "costo_p999": q[2]})
for k, v in dq.items():
    print(f"{k:24s} {v}")
resumen["eventos"] = {k: (str(v) if isinstance(v, datetime) else v) for k, v in dq.items()}
resumen["eventos"]["filas"] = n_ev

event_id_distintos       43200
event_id_nulos           0
timestamp_no_parseable   0
ts_min_utc               2025-07-03 00:02:00
ts_max_utc               2025-08-31 23:58:00
dias_con_eventos         60
value_nulos              877
unit_nulos               2075
unit_nulo_con_value      2038
costo_menor_a_-0.01      211
costo_negativo           216
costo_min                -154.4608
costo_max                317.4308
event_id_duplicados      0
costo_p50                1.0044
costo_p99                16.7175
costo_p999               133.141


In [8]:
print("Evolución de esquema (schema_version):")
ev.groupBy("schema_version").agg(
    F.count("*").alias("eventos"), F.min("event_ts").alias("desde"), F.max("event_ts").alias("hasta"),
    F.sum(F.col("carbon_kg").isNotNull().cast("int")).alias("con_carbon_kg"),
    F.sum(F.col("genai_tokens").isNotNull().cast("int")).alias("con_genai_tokens"),
).orderBy("schema_version").show(truncate=False)

print("genai_tokens presente por servicio:")
ev.where(F.col("genai_tokens").isNotNull()).groupBy("service").count().show()

print("metric × unit (incluye unit nulo):")
ev.groupBy("metric").pivot("unit").count().na.fill(0).show()

print("Eventos por servicio y región:")
ev.groupBy("service").count().orderBy(F.desc("count")).show()
ev.groupBy("region").count().orderBy(F.desc("count")).show()
resumen["schema_version"] = {str(r["schema_version"]): r["count"] for r in ev.groupBy("schema_version").count().collect()}

Evolución de esquema (schema_version):


+--------------+-------+-------------------+-------------------+-------------+----------------+
|schema_version|eventos|desde              |hasta              |con_carbon_kg|con_genai_tokens|
+--------------+-------+-------------------+-------------------+-------------+----------------+
|1             |10800  |2025-07-03 00:02:00|2025-07-17 23:56:00|0            |0               |
|2             |32400  |2025-07-18 00:01:00|2025-08-31 23:58:00|32400        |3132            |
+--------------+-------+-------------------+-------------------+-------------+----------------+

genai_tokens presente por servicio:


+-------+-----+
|service|count|
+-------+-----+
|  genai| 3132|
+-------+-----+

metric × unit (incluye unit nulo):


+----------------+----+-----+--------+-----+
|          metric|null|count|gb_hours|hours|
+----------------+----+-----+--------+-----+
|       cpu_hours| 505|    0|       0|10187|
|storage_gb_hours| 654|    0|   12342|    0|
|        requests| 916|18596|       0|    0|
+----------------+----+-----+--------+-----+

Eventos por servicio y región:


+----------+-----+
|   service|count|
+----------+-----+
|   compute|12498|
|   storage| 7614|
|  database| 7419|
|networking| 6921|
| analytics| 4590|
|     genai| 4158|
+----------+-----+



+------------+-----+
|      region|count|
+------------+-----+
|ap-northeast| 7773|
|     us-east| 7566|
|     us-west| 6384|
|     sa-east| 6270|
|    ap-south| 5943|
|     eu-west| 4854|
|  eu-central| 4410|
+------------+-----+



In [9]:
# Tipo JSON original de `value` y `carbon_kg` (la inferencia de Spark los unifica)
raw = spark.read.text(events_files)
tipos = raw.select(
    F.when(F.col("value").rlike(r'"value":\s*null'), "null")
     .when(F.col("value").rlike(r'"value":\s*"'), "string")
     .otherwise("numérico").alias("tipo_value"),
    F.when(~F.col("value").contains('"carbon_kg"'), "ausente")
     .when(F.col("value").rlike(r'"carbon_kg":\s*-?\d+[,}]'), "entero")
     .otherwise("decimal").alias("tipo_carbon_kg"))
tipos.groupBy("tipo_value").count().orderBy("tipo_value").show()
tipos.groupBy("tipo_carbon_kg").count().orderBy("tipo_carbon_kg").show()
resumen["tipo_json_value"] = {r[0]: r[1] for r in tipos.groupBy("tipo_value").count().collect()}

no_casteables = ev.where(F.col("value").isNotNull() & F.col("value").cast("double").isNull()).count()
print("value no casteable a double:", no_casteables)
resumen["value_no_casteable"] = no_casteables

+----------+-----+
|tipo_value|count|
+----------+-----+
|      null|  877|
|  numérico|41014|
|    string| 1309|
+----------+-----+



+--------------+-----+
|tipo_carbon_kg|count|
+--------------+-----+
|       ausente|10800|
|       decimal|29762|
|        entero| 2638|
+--------------+-----+



value no casteable a double: 0


In [10]:
# Integridad referencial de eventos contra los maestros
res_df = dfs["resources"].select("resource_id", F.col("service").alias("svc_recurso"))
ref = {
    "org_id huérfanos": ev.join(orgs, "org_id", "left_anti").count(),
    "resource_id huérfanos": ev.join(res_df, "resource_id", "left_anti").count(),
    "service distinto al del recurso": ev.join(res_df, "resource_id").where(F.col("service") != F.col("svc_recurso")).count(),
}
for k, v in ref.items():
    print(f"{k:35s} {v}")
resumen["eventos_integridad"] = ref

org_id huérfanos                    0
resource_id huérfanos               0
service distinto al del recurso     0


## 4. Orden temporal de los micro-lotes (impacto en watermark)

Structured Streaming procesa los archivos en orden. Si cada archivo cubre todo el período, un watermark por *event-time* avanzaría al final del período en el primer micro-lote y los siguientes eventos serían "tardíos". Se mide el rango temporal por archivo y se **simula** cuántos eventos descartaría una deduplicación con estado (`dropDuplicatesWithinWatermark`) con `maxFilesPerTrigger=10` (12 micro-lotes).

In [11]:
por_archivo = (ev.groupBy("source_file")
                 .agg(F.count("*").alias("eventos"), F.min("event_ts").alias("ts_min"), F.max("event_ts").alias("ts_max"))
                 .withColumn("dias_cubiertos", F.datediff("ts_max", "ts_min") + 1)
                 .orderBy("source_file"))
por_archivo.show(5, truncate=False)
por_archivo.select(F.min("dias_cubiertos"), F.max("dias_cubiertos"), F.min("eventos"), F.max("eventos")).show()

+----------------------+-------+-------------------+-------------------+--------------+
|source_file           |eventos|ts_min             |ts_max             |dias_cubiertos|
+----------------------+-------+-------------------+-------------------+--------------+
|events_part_0000.jsonl|360    |2025-07-03 00:08:00|2025-08-31 16:32:00|60            |
|events_part_0001.jsonl|360    |2025-07-03 02:38:00|2025-08-31 23:51:00|60            |
|events_part_0002.jsonl|360    |2025-07-03 14:16:00|2025-08-31 20:47:00|60            |
|events_part_0003.jsonl|360    |2025-07-03 00:48:00|2025-08-31 23:51:00|60            |
|events_part_0004.jsonl|360    |2025-07-03 10:43:00|2025-08-31 20:40:00|60            |
+----------------------+-------+-------------------+-------------------+--------------+
only showing top 5 rows



+-------------------+-------------------+------------+------------+
|min(dias_cubiertos)|max(dias_cubiertos)|min(eventos)|max(eventos)|
+-------------------+-------------------+------------+------------+
|                 60|                 60|         360|         360|
+-------------------+-------------------+------------+------------+



In [12]:
FILES_PER_TRIGGER = 10
from pyspark.sql import Window
# micro_lote k = archivos [10k, 10k+9] por nombre. El file source de Spark toma los archivos por fecha de
# modificación; en esta copia ese orden coincide con el nombre. Como cada archivo cubre los 60 días,
# el resultado es prácticamente el mismo con cualquier orden.
evl = ev.withColumn("micro_lote", F.floor((F.dense_rank().over(Window.orderBy("source_file")) - 1) / FILES_PER_TRIGGER))
max_lote = evl.groupBy("micro_lote").agg(F.max("event_ts").cast("long").alias("max_ts"))
# watermark vigente al procesar el lote k = max(event_ts de los lotes anteriores) - umbral
previos = Window.orderBy("micro_lote").rowsBetween(Window.unboundedPreceding, -1)
max_previo = max_lote.withColumn("max_previo", F.max("max_ts").over(previos)).select("micro_lote", "max_previo")
evw = evl.join(max_previo, "micro_lote").cache()

simulacion = []
for umbral_h in [1, 24, 24 * 7, 24 * 30, 24 * 61]:
    descartados = evw.where(F.col("event_ts").cast("long") < F.col("max_previo") - umbral_h * 3600).count()
    simulacion.append({"umbral_horas": umbral_h, "descartados": descartados, "pct": round(100 * descartados / n_ev, 1)})
for s in simulacion:
    print(f"watermark {s['umbral_horas']:5d} h → descartaría {s['descartados']:6d} eventos ({s['pct']} %)")
resumen["simulacion_watermark_event_time"] = simulacion

watermark     1 h → descartaría  39566 eventos (91.6 %)
watermark    24 h → descartaría  38935 eventos (90.1 %)
watermark   168 h → descartaría  34983 eventos (81.0 %)
watermark   720 h → descartaría  19758 eventos (45.7 %)
watermark  1464 h → descartaría      0 eventos (0.0 %)


## 5. Resumen y evidencia

In [13]:
out = os.path.join(EVIDENCE_DIR, "perfil_landing.json")
with open(out, "w", encoding="utf-8") as fh:
    json.dump(resumen, fh, ensure_ascii=False, indent=2, default=str)
print("Evidencia escrita en", out)
spark.stop()

Evidencia escrita en ../evidence/entrega1/perfil_landing.json
